Carlos Matias Jose Silva Timmermann

22.681.769-7

In [122]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [123]:
rng = np.random.default_rng(seed = 52777)

Parte 1

In [124]:
#Constantes fisicas

sigma_sb = 5.670374419e-8
r_sol = 6.9634e8
l_sol = 3.828e26
pc_a_m = 3.085677581491367e16

In [125]:
#Generacion de catalogo

n_estrellas = 100
R = rng.uniform(0.5, 5, n_estrellas)
D = rng.uniform(50, 500, n_estrellas)
T_eff = rng.uniform(3000, 10000, n_estrellas)

df = pd.DataFrame({
    'ID_estrella' : [f'Estrella_{i + 1}' for i in range(n_estrellas)],
    'Radio_solar' : R,
    'Distancia_pc' : D,
    'Temperatura_K' : T_eff
})

In [ ]:
#Visualizacion de las primeras 10 filas

print("Primeras 10 filas del catalogo : ")
print(df.head(10))

In [ ]:
#Estadisticas de lo simulado

print("Estadisticas de las propiedades simuladas : ")
print(df.describe())

Parte 2

In [128]:
#Calculo de luminosidad

r_m = df['Radio_solar'] * r_sol
l_w = 4 * np.pi * (r_m**2) * sigma_sb * (df['Temperatura_K']**4)
df['Luminosidad_solar'] = l_w / l_sol


In [129]:
#Calculo del flujo

d_m = df['Distancia_pc'] * pc_a_m
df['Flujo'] = l_w / (4 * np.pi * (d_m**2))

In [ ]:
#Printeo de la inclusion de columnas al DataFrame

print("Muestra del catalogo con luminosidad y flujo")
print(df[['ID_estrella', 'Radio_solar', 'Distancia_pc', 'Temperatura_K', 'Luminosidad_solar', 'Flujo']].head())

In [131]:
#Estrellas con valores extremos

max_lum = df.loc[df['Luminosidad_solar'].idxmax()]
min_lum = df.loc[df['Luminosidad_solar'].idxmin()]
max_flujo = df.loc[df['Flujo'].idxmax()]
min_flujo = df.loc[df['Flujo'].idxmin()]

In [ ]:
#Printeo de valores extremos

print("Estrellas con valores extremos")
print(f"Estrella con mayor Luminosidad : {max_lum['ID_estrella']} {max_lum['Luminosidad_solar']:.4f}")
print(f"Estrella con menor Luminosidad : {min_lum['ID_estrella']} {min_lum['Luminosidad_solar']:.4f}")
print(f"Estrella con mayor Flujo       : {max_flujo['ID_estrella']} {max_flujo['Flujo']:.4e}")
print(f"Estrella con menor Flujo       : {min_flujo['ID_estrella']} {min_flujo['Flujo']:.4e}")

Parte 3

In [133]:
#Limite de deteccion

Flujo_limite = 2e-10

In [134]:
#Se agrega una columna al catalogo

df['Clasificacion'] = np.where(df['Flujo'] >= Flujo_limite, 'Detectada', 'No detectada')

In [135]:
#Conteo por categoria y porcentaje

conteo_categorias = df['Clasificacion'].value_counts()
porcentaje_detectadas = (df['Clasificacion'] == 'Detectada').mean() * 100

In [ ]:
#Printeo de conteo por categoria y porcentaje

print("Resultados de clasificacion : ")
print(f"Conteo por categoria : {conteo_categorias}")
print(f"Porcentaje de estrellas detectadas : {porcentaje_detectadas:.2f} %")

In [137]:
#Promedio de flujos, distancia y temperatura por categoria

promedios = df.groupby('Clasificacion')[['Flujo', 'Distancia_pc', 'Temperatura_K']].mean()
promedios.columns = ['Flujo Promedio', 'Distancia Promedio', 'Temperatura K Promedio']

In [ ]:
#Printeo del promedio de flujos, distancia y temperatura por categoria

print("Promedios estadísticos según categoría :")
print(promedios)

Parte 4

In [ ]:
#Histogramas de distribuciones fisicas

fig, axes = plt.subplots(3, 2, figsize = (13, 14))
fig.suptitle('Distribución de propiedades fisicas en la muestra', fontsize = 16, fontweight = 'bold')

#Histogtama de temperatura efectiva

axes[0, 0].hist(df['Temperatura_K'], bins = 12, color = 'blue', edgecolor = 'black', alpha = 0.7)
axes[0, 0].set_title('Distribución de temperaturas')
axes[0, 0].set_xlabel('Temperatura efectiva')
axes[0, 0].set_ylabel('Frecuencia')

#Histograma del radio estelar

axes[0, 1].hist(df['Radio_solar'], bins = 12, color = 'orange', edgecolor = 'black', alpha = 0.7)
axes[0, 1].set_title('Distribución de radios estelares')
axes[0, 1].set_xlabel('Radio estelar')
axes[0, 1].set_ylabel('Frecuencia')

#Histograma de la distancia

axes[1, 0].hist(df['Distancia_pc'], bins = 12, color = 'green', edgecolor = 'black', alpha = 0.7)
axes[1, 0].set_title('Distribución de distancias')
axes[1, 0].set_xlabel('Distancia')
axes[1, 0].set_ylabel('Frecuencia')


#Histograma de la luminosidad

axes[1, 1].hist(df['Luminosidad_solar'], bins = 12, color = 'purple', edgecolor = 'black', alpha = 0.7)
axes[1, 1].set_title('Distribución de luminosidades')
axes[1, 1].set_xlabel('Luminosidad')
axes[1, 1].set_ylabel('Frecuencia')

#Histograma del flujo

axes[2, 0].hist(df['Flujo'], bins = 12, color = 'red', edgecolor = 'black', alpha = 0.7)
axes[2, 0].set_title('Distribución de flujos')
axes[2, 0].set_xlabel('Flujo')
axes[2, 0].set_ylabel('Frecuencia')

#El ultimo panel se esconde y mostramos la grafica

axes[2, 1].axis('off')

plt.tight_layout()
plt.subplots_adjust(top = 1)
plt.savefig('histogramas_propiedades.png', dpi = 300)
plt.show()

In [ ]:
#Dispersion de flujo vs dispercion de la distancia

plt.figure(figsize = (9, 6))

#variables detectadas y no_detectadas

detectadas = df[df['Clasificacion'] == 'Detectada']
no_detectadas = df[df['Clasificacion'] == 'No detectada']

#Grafico de puntos por categoria de deteccion

plt.scatter(no_detectadas['Distancia_pc'], no_detectadas['Flujo'], color = 'gray', alpha = 0.6, label = 'No detectada', s = 50)
plt.scatter(detectadas['Distancia_pc'], detectadas['Flujo'], color = 'cyan', alpha = 0.9, label = 'Detectada', s = 70, edgecolors = 'black')

#Linea horizontal del límite de detección del telescopio

plt.axhline(Flujo_limite, color = 'brown', linestyle = '--', linewidth = 2, label = f'Límite de detección')

#Escala logaritmica para el eje del flujo

plt.yscale('log')

#Configuracion final

plt.title('Flujo en funcion de la distancia', fontsize = 14, fontweight = 'bold')
plt.xlabel('Distancia', fontsize = 12)
plt.ylabel('Flujo Observado (escala logaritmica)', fontsize = 12)
plt.legend(frameon = True, loc = 'upper right')
plt.grid(True, which = "both", ls = "--", alpha = 0.5)
plt.tight_layout()
plt.savefig('flujo_vs_distancia.png', dpi = 300)
plt.show()

In [ ]:
#Diagrama de la luminosidad vs la temperatura

plt.figure(figsize = (9, 6))

scatter = plt.scatter(df['Temperatura_K'], df['Luminosidad_solar'], c = df['Radio_solar'], cmap = 'viridis', s = 60, alpha = 0.85)

#Barra indicando el radio estelar

cbar = plt.colorbar(scatter)
cbar.set_label('Radio estelar', fontsize = 12)

#Escala logaritmica

plt.yscale('log')

#Se invierte el eje de la temperatura

plt.gca().invert_xaxis()

#Configuracion final

plt.title('Luminosidad en función de la temperatura', fontsize = 14, fontweight = 'bold')
plt.xlabel('Temperatura (eje Invertido)', fontsize = 12)
plt.ylabel('Luminosidad (escala logaritmica)', fontsize = 12)
plt.grid(True, which = "both", ls = "--", alpha = 0.5)
plt.tight_layout()
plt.savefig('luminosidad_vs_temperatura.png', dpi=300)
plt.show()